# Can we predict a warranty claim within a year?
Follows `protocol.yaml`: fixed holdout (`holdout.csv`), stratified 5-fold CV on training rows, selection by mean CV ROC AUC, one holdout score for the chosen candidate.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score

df = pd.read_csv('data.csv')
assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='25552b73fd810f7dfa81a14849736291d84f91da2f8b5a58bcfaf0cc018962f4'
holdout_idx = pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==7000 and len(holdout_idx)==1400
target='claim_within_year'
train_idx = np.setdiff1d(np.arange(len(df)), holdout_idx)
train, hold = df.iloc[train_idx], df.iloc[holdout_idx]
print(len(train), len(hold), train[target].mean(), hold[target].mean())

## Features
`device_serial` is excluded (identifier). The other four columns (sales channel, device age, price, reported daily use) are all known at sale/registration, i.e. before the outcome.

In [ ]:
cat=['channel']; num=['device_age_years','price_eur','daily_hours']
features=cat+num
X, y = train[features], train[target]
print(train.groupby('channel')[target].mean()); print(train[num].describe())

## Candidates (declared before any scoring)
Preprocessing lives inside each pipeline, so it is fit within CV folds.

In [ ]:
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),cat),
                              ('n',StandardScaler() if scale else 'passthrough',num)])
candidates = {
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'logreg_C0.1': Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'rf_leaf20': Pipeline([('p',prep(False)),('m',RandomForestClassifier(n_estimators=300,min_samples_leaf=20,random_state=0,n_jobs=-1))]),
 'hgb_small': Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,random_state=0))]),
}
print(list(candidates))

## Cross-validation on training rows only

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
cv_scores={n:cross_val_score(m,X,y,cv=cv,scoring='roc_auc').tolist() for n,m in candidates.items()}
summary=pd.DataFrame({n:{'mean':np.mean(s),'std':np.std(s)} for n,s in cv_scores.items()}).T
print(summary.sort_values('mean',ascending=False))
chosen=summary['mean'].idxmax(); print('chosen:',chosen)

## Refit chosen candidate on all training rows; score once on the holdout

In [ ]:
final=candidates[chosen].fit(X,y)
holdout_auc=roc_auc_score(hold[target], final.predict_proba(hold[features])[:,1])
print('holdout roc_auc:',holdout_auc)

In [ ]:
# (train, validation) 0-based data.csv row positions for each CV fold
cv_folds=[[[int(train_idx[i]) for i in tr],[int(train_idx[i]) for i in va]] for tr,va in cv.split(X,y)]
evidence={'features':features,'candidates':list(candidates),'cv_folds':cv_folds,'cv_scores':cv_scores,
 'chosen':chosen,'holdout_rows':[int(i) for i in holdout_idx],'holdout_scores':{chosen:holdout_auc}}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above: the holdout AUC is the headline estimate of how well claims within a year can be predicted from these four sale-time features (0.5 = no better than chance).